# Statistical POS-Tagged Parser using the Penn Treebank

This notebook builds a statistical POS tagger and a PCFG parser from the NLTK Penn Treebank, then evaluates them on 150 sentences collected from the Daily Mirror online newspaper.

## 1. Setup and Corpus Loading

This section imports the required libraries, loads the Penn Treebank corpus, and reports basic statistics about it.

### 1.1 Import Libraries

[ Write your explanation here ]

In [1]:
import nltk
from nltk.corpus import treebank
from collections import Counter

# download the corpus the first time only
try:
    nltk.data.find("corpora/treebank")
except LookupError:
    nltk.download("treebank")

### 1.2 Load the NLTK Penn Treebank (Labeled)

Load the corpus in two forms:
- tagged word/tag pairs for the tagger.
- parsed trees for the PCFG parser.

In [2]:
tagged_sents = treebank.tagged_sents()   # for the TAGGER  (word/tag pairs)
parsed_sents = treebank.parsed_sents()   # for the PARSER  (structure trees)

print("First tagged sentence (first 6 tokens):")
print(tagged_sents[0][:6])

First tagged sentence (first 6 tokens):
[('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ')]


Visualize the structure (tree form) of the first sentence

In [3]:
parsed_sents[0].pretty_print()

                                                     S                                                                         
                         ____________________________|_______________________________________________________________________   
                        |                                               VP                                                   | 
                        |                        _______________________|___                                                 |  
                      NP-SBJ                    |                           VP                                               | 
         _______________|___________________    |     ______________________|______________________________________          |  
        |          |              ADJP      |   |    |        |                PP-CLR                              |         | 
        |          |           ____|____    |   |    |        |          ________|_________          

### 1.3 Corpus Statistics

Compute basic statistics (sentences, tokens, distinct words and tags) to understand the training data. The small size (~3,900 sentences) is relevant to unknown-word errors discussed later.

In [4]:
n_sents  = len(tagged_sents)
tokens   = [tok for s in tagged_sents for tok in s]
n_tokens = len(tokens)
tagset   = {t for _, t in tokens}
vocab    = {w.lower() for w, _ in tokens}

print(f"Sentences            : {n_sents}")
print(f"Tokens (words)       : {n_tokens}")
print(f"Distinct word types  : {len(vocab)}")
print(f"Distinct POS tags    : {len(tagset)}")
print(f"Avg sentence length  : {n_tokens/n_sents:.1f} words")

Sentences            : 3914
Tokens (words)       : 100676
Distinct word types  : 11387
Distinct POS tags    : 46
Avg sentence length  : 25.7 words


## 2. Training the Statistical POS Tagger (HMM)

This section trains the HMM tagger by building its two probability tables from the corpus: transition probabilities (grammar flow) and emission probabilities (word–tag association). These tables are the tagger's learned knowledge, used later with the Viterbi algorithm.

### 2.1 Transition Probabilities

Bigram tag-to-tag transition counts are tallied across the corpus and normalized by maximum likelihood estimation into conditional probabilities P(tag | previous tag), capturing the sequential syntactic dependencies between adjacent tags.

In [5]:
from collections import defaultdict

# count how often each tag follows the previous tag
transition_counts = defaultdict(Counter)   # transition_counts[prev][curr] = count
tag_unigram = Counter()                     # how often each tag occurs overall

for sent in tagged_sents:
    prev = "<s>"                            # sentence-start marker
    for word, tag in sent:
        transition_counts[prev][tag] += 1
        tag_unigram[tag] += 1
        prev = tag

# turn counts into probabilities:  P(curr | prev) = count(prev, curr) / count(prev)
transition_prob = defaultdict(dict)
for prev, nexts in transition_counts.items():
    total = sum(nexts.values())
    for curr, c in nexts.items():
        transition_prob[prev][curr] = c / total

# sanity check: what most commonly follows a determiner (DT)?
print("Most likely tags after DT:")
for tag, p in sorted(transition_prob["DT"].items(), key=lambda x: -x[1])[:5]:
    print(f"  P({tag:4} | DT) = {p:.3f}")

Most likely tags after DT:
  P(NN   | DT) = 0.471
  P(JJ   | DT) = 0.203
  P(NNP  | DT) = 0.129
  P(NNS  | DT) = 0.077
  P(CD   | DT) = 0.023


Transition table

In [6]:
# Understanding the transition table: how grammar "flows"
print("Table size:")
print(f"  {len(transition_prob)} 'previous-tag' rows (each row is a full probability distribution)\n")

# look at several different previous-tags to see the grammar patterns
for prev in ["<s>", "DT", "JJ", "IN", "MD"]:
    top = sorted(transition_prob[prev].items(), key=lambda x: -x[1])[:3]
    shown = ", ".join(f"{tag} ({p:.2f})" for tag, p in top)
    print(f"After {prev:4} -> {shown}")

Table size:
  47 'previous-tag' rows (each row is a full probability distribution)

After <s>  -> DT (0.23), NNP (0.20), IN (0.13)
After DT   -> NN (0.47), JJ (0.20), NNP (0.13)
After JJ   -> NN (0.45), NNS (0.24), JJ (0.06)
After IN   -> DT (0.32), NNP (0.15), NN (0.11)
After MD   -> VB (0.82), RB (0.16), -NONE- (0.01)


### 2.2 Emission Probabilities

Tag-conditioned lexical frequencies are computed from the tagged corpus and normalized via maximum likelihood estimation into emission probabilities P(word | tag), quantifying the lexical generation likelihood of each observation given its category.

In [7]:
# count how often each tag produces each word
emission_counts = defaultdict(Counter)      # emission_counts[tag][word] = count

for sent in tagged_sents:
    for word, tag in sent:
        emission_counts[tag][word.lower()] += 1

# turn counts into probabilities:  P(word | tag) = count(tag, word) / count(tag)
emission_prob = defaultdict(dict)
for tag, words in emission_counts.items():
    total = sum(words.values())
    for word, c in words.items():
        emission_prob[tag][word] = c / total

# sanity check: the most likely words for a singular noun (NN)
print("Most likely words for NN:")
for word, p in sorted(emission_prob["NN"].items(), key=lambda x: -x[1])[:5]:
    print(f"  P({word!r:12} | NN) = {p:.4f}")

Most likely words for NN:
  P('%'          | NN) = 0.0338
  P('company'    | NN) = 0.0197
  P('year'       | NN) = 0.0163
  P('market'     | NN) = 0.0135
  P('trading'    | NN) = 0.0112


Emission table

In [8]:
# Understanding the emission table: which words belong to which tag
print("Table size:")
print(f"  {len(emission_prob)} tag rows; e.g. NN holds {len(emission_prob['NN'])} distinct words\n")

# top words for several different tags
for tag in ["NN", "VBD", "JJ", "IN", "PRP"]:
    top = sorted(emission_prob[tag].items(), key=lambda x: -x[1])[:4]
    shown = ", ".join(f"{w} ({p:.3f})" for w, p in top)
    print(f"{tag:4} -> {shown}")

Table size:
  46 tag rows; e.g. NN holds 2506 distinct words

NN   -> % (0.034), company (0.020), year (0.016), market (0.014)
VBD  -> said (0.202), was (0.121), were (0.065), had (0.053)
JJ   -> new (0.029), other (0.024), last (0.017), many (0.017)
IN   -> of (0.236), in (0.177), for (0.086), that (0.052)
PRP  -> it (0.336), he (0.177), they (0.153), i (0.066)


### 2.3 Smoothing and Unknown-Word Handling

Add-one smoothing is applied to eliminate zero probabilities, and out-of-vocabulary words are handled using a data-driven suffix model with orthographic cues, toggleable via a switch.

**Precompute totals + vocabulary + the switch**

Per-tag totals and the training vocabulary are precomputed for efficiency, and a boolean switch (USE_CLUES) is defined to toggle the unknown-word mechanism for controlled comparison.

In [9]:
import math

# SWITCH: turn the smart unknown-word clue mechanism ON or OFF
USE_CLUES = True     # True = learned suffix + digit + capitalization clues
                     # False = plain uniform fallback (basic baseline)

# precompute totals once (for speed)
all_tags = list(tag_unigram.keys())
num_tags = len(all_tags)
transition_totals = {p: sum(c.values()) for p, c in transition_counts.items()}
emission_totals   = {t: sum(c.values()) for t, c in emission_counts.items()}

# vocabulary + word frequencies (to detect unknown/rare words)
known_words = set()
word_freq   = Counter()
for sent in tagged_sents:
    for word, tag in sent:
        known_words.add(word.lower())
        word_freq[word.lower()] += 1
V = len(known_words)

print("Vocabulary size:", V, "| tags:", num_tags)

Vocabulary size: 11387 | tags: 46


**Learn the suffix --> tag patterns (unknown-word model)**

A suffix-to-tag distribution is learned from low-frequency training words to infer the likely categories of unseen words from their endings.

In [10]:
# LEARN suffix -> tag patterns from RARE words (proxies for unknowns)
MAX_SUFFIX, RARE = 3, 2
suffix_tag_counts = defaultdict(Counter)
for sent in tagged_sents:
    for word, tag in sent:
        wl = word.lower()
        if word_freq[wl] <= RARE:                      # rare words behave like unknowns
            for k in range(1, MAX_SUFFIX + 1):
                if len(wl) >= k:
                    suffix_tag_counts[wl[-k:]][tag] += 1

def suffix_scores(word):
    """Learned tag distribution for a word's ending (backing off 3->2->1 chars)."""
    wl = word.lower()
    for k in range(MAX_SUFFIX, 0, -1):
        if wl[-k:] in suffix_tag_counts:
            c = suffix_tag_counts[wl[-k:]]
            tot = sum(c.values())
            return {tg: n / tot for tg, n in c.items()}
    return None

print("Suffix patterns learned:", len(suffix_tag_counts))

Suffix patterns learned: 2311


**The smoothed probability lookups**

Add-one smoothing is applied to the transition and emission lookups; unknown words are scored by the learned clues or, when disabled, by a uniform fallback.

In [11]:
FLOOR = 1e-8

# smoothed transition probability
def get_transition(prev, curr):
    counts = transition_counts.get(prev, {})
    total  = transition_totals.get(prev, 0)
    return (counts.get(curr, 0) + 1) / (total + num_tags)

# smoothed emission, with clues for unknown words
def get_emission(word, tag):
    wl = word.lower()
    if wl in known_words:                              # seen in training -> smoothed count
        return (emission_counts[tag].get(wl, 0) + 1) / (emission_totals.get(tag, 0) + V)

    # unknown word
    if not USE_CLUES:
        return 1.0 / (emission_totals.get(tag, 0) + V)   # plain uniform fallback

    dist  = suffix_scores(word)                          # learned suffix distribution
    score = (1.0 / num_tags) if dist is None else dist.get(tag, 0.0)
    if any(ch.isdigit() for ch in word):                 # digit -> number
        score = 0.9 if tag == "CD" else score * 0.1
    if word[0].isupper() and tag in ("NNP", "NNPS"):     # capital -> proper noun
        score += 0.5
    return max(score, FLOOR)

print("Smoothed lookups ready.")

Smoothed lookups ready.


### 2.4 Viterbi Decoding
The Viterbi algorithm is implemented in log-space to decode the most probable tag sequence using dynamic programming with back-pointers.

In [12]:
def viterbi(sentence):
    """sentence: list of word strings. Returns list of (word, tag)."""
    if not sentence:
        return []
    Vt = [{}]; bp = [{}]                                 # scores + back-pointers

    # first word: transition from the <s> start marker
    for tag in all_tags:
        Vt[0][tag] = math.log(get_transition("<s>", tag)) + math.log(get_emission(sentence[0], tag))
        bp[0][tag] = "<s>"

    # remaining words
    for i in range(1, len(sentence)):
        Vt.append({}); bp.append({})
        for tag in all_tags:
            best_prev, best = None, float("-inf")
            for pt in all_tags:                          # best way to reach `tag`
                s = Vt[i-1][pt] + math.log(get_transition(pt, tag))
                if s > best:
                    best, best_prev = s, pt
            Vt[i][tag] = best + math.log(get_emission(sentence[i], tag))
            bp[i][tag] = best_prev

    # trace back the winning path
    last = max(Vt[-1], key=Vt[-1].get)
    tags = [last]
    for i in range(len(sentence) - 1, 0, -1):
        last = bp[i][last]
        tags.insert(0, last)
    return list(zip(sentence, tags))

# test the finished tagger
for s in ["The winning team wowed in the stadium",
          "Officials announced a new economic policy"]:
    print(s)
    for word, tag in viterbi(s.split()):
        print(f"   {word:12} -> {tag}")
    print()

The winning team wowed in the stadium
   The          -> DT
   winning      -> JJ
   team         -> NN
   wowed        -> VBD
   in           -> IN
   the          -> DT
   stadium      -> NN

Officials announced a new economic policy
   Officials    -> NNS
   announced    -> VBD
   a            -> DT
   new          -> JJ
   economic     -> JJ
   policy       -> NN



### 2.5 Effect of the clue mechanism
A controlled comparison illustrates the effect of the unknown-word clue mechanism on the tags assigned to unseen words.

In [13]:
# --- compare tagging WITH vs WITHOUT the clue mechanism ---
test = "Colombo hosted a thrilling cricketing festival".split()

USE_CLUES = False
off = viterbi(test)
USE_CLUES = True
on  = viterbi(test)

print(f"{'word':12} {'CLUES OFF':10} {'CLUES ON':10}")
print("-" * 34)
for (w, t_off), (_, t_on) in zip(off, on):
    mark = "" if t_off == t_on else "  <-- changed"
    print(f"{w:12} {t_off:10} {t_on:10}{mark}")

USE_CLUES = True   # leave it ON for the rest of the notebook

word         CLUES OFF  CLUES ON  
----------------------------------
Colombo      PRP        NNP         <-- changed
hosted       VBD        ,           <-- changed
a            DT         DT        
thrilling    NN         JJ          <-- changed
cricketing   .          NN          <-- changed
festival     ''         NN          <-- changed


### 2.6 Save the trained tagger to disk

The trained tagger's learned parameters are serialized to disk so the model can be reused without retraining and submitted as a deliverable.

**Save the tagger**

The transition and emission counts, vocabulary, word frequencies, and learned suffix patterns are bundled and serialized to models/hmm_tagger.pkl using pickle.

In [14]:
import pickle, os

os.makedirs("models", exist_ok=True)   # make the models/ folder if needed

# bundle everything the tagger needs into one dictionary
tagger_model = {
    "transition_counts": {p: dict(c) for p, c in transition_counts.items()},
    "emission_counts":   {t: dict(c) for t, c in emission_counts.items()},
    "tag_unigram":       dict(tag_unigram),
    "known_words":       known_words,
    "word_freq":         dict(word_freq),
    "suffix_tag_counts": {s: dict(c) for s, c in suffix_tag_counts.items()},
}

with open("models/hmm_tagger.pkl", "wb") as f:
    pickle.dump(tagger_model, f)

print("Saved -> models/hmm_tagger.pkl")
print("File size:", round(os.path.getsize("models/hmm_tagger.pkl")/1024, 1), "KB")

Saved -> models/hmm_tagger.pkl
File size: 464.6 KB


**Verify it reloads**

The serialized model is reloaded and its contents verified to confirm the save is complete and intact.

In [15]:
# reload and confirm the saved model works
with open("models/hmm_tagger.pkl", "rb") as f:
    loaded = pickle.load(f)

print("Reloaded successfully.")
print("  tags saved      :", len(loaded["tag_unigram"]))
print("  vocabulary saved:", len(loaded["known_words"]))
print("  suffix patterns :", len(loaded["suffix_tag_counts"]))

Reloaded successfully.
  tags saved      : 46
  vocabulary saved: 11387
  suffix patterns : 2311


## 3. Building the PCFG Parser

[ Write your explanation here ]

### 3.1 Extract Productions from the Parsed Trees

[ Write your explanation here ]

### 3.2 Convert Trees to Chomsky Normal Form

[ Write your explanation here ]

### 3.3 Induce the PCFG

[ Write your explanation here ]

### 3.4 Save the PCFG Parser Data

[ Write your explanation here ]

### 3.5 Parsing Sanity Check

[ Write your explanation here ]

## 4. Preparing the 150 Daily Mirror Sentences

[ Write your explanation here ]

### 4.1 Load the Raw Sentences

[ Write your explanation here ]

### 4.2 Penn Treebank Tokenization

[ Write your explanation here ]

### 4.3 Load the Gold-Standard Tags

[ Write your explanation here ]

## 5. Tagging and Parsing the 150 Sentences

[ Write your explanation here ]

### 5.1 Tag with the Trained Model

[ Write your explanation here ]

### 5.2 Parse a Sample of Sentences

[ Write your explanation here ]

### 5.3 Save the Predictions

[ Write your explanation here ]

## 6. Evaluation

[ Write your explanation here ]

### 6.1 Overall Accuracy and Baseline

[ Write your explanation here ]

### 6.2 Per-Tag Precision, Recall and F1

[ Write your explanation here ]

### 6.3 Confusion Matrix

[ Write your explanation here ]

### 6.4 Out-of-Vocabulary Analysis

[ Write your explanation here ]

### 6.5 Save the Results

[ Write your explanation here ]

## 7. Results Summary and Error Analysis

[ Write your explanation here ]